# Tokenizers & English Morphology — Onboarding

**Question:** Do tokenizers split words at the right place less often when adding a suffix changes the spelling?

Run the cells top to bottom. There are **three TODOs** (sections 3, 4 and 6).

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Find the repo root so this works from starter/ or submissions/<username>/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "onboarding").is_dir())
sys.path.append(str(ROOT / "src"))

from onboarding.data import load_morphynet, prepare_suffix_data, balanced_sample
from onboarding.tokenizers import load_tokenizers, tokenize_with_offsets

## 1. Load the data

Each row has a `source` word, a `morpheme` (the suffix), and the `target` word they form. `suffix_boundary` is the character position where the suffix starts in `target`.

`group` is `concatenative` when `target == source + morpheme` (slow + ly → slowly) and `rewritten` otherwise (happy + ness → happiness).

In [ ]:
suffixes = prepare_suffix_data(load_morphynet())
print(suffixes["group"].value_counts())

cols = ["source", "morpheme", "target", "suffix_boundary"]
display(suffixes[suffixes["group"] == "concatenative"][cols].sample(5, random_state=0))
display(suffixes[suffixes["group"] == "rewritten"][cols].sample(5, random_state=0))

The `rewritten` label comes from a rough rule, so it isn't perfect ground truth. Note anything that looks odd.

Next, sample 500 words per group. The seed is fixed so everyone gets the same sample.

In [ ]:
sample = balanced_sample(suffixes, n_per_group=500, random_state=42)
sample["group"].value_counts()

## 2. See how tokenizers split words

We compare `bert-base-uncased` and `gpt2`. Each token comes with an **offset** `(start, end)`, which gives the characters of the word it covers.

In [ ]:
tokenizers = load_tokenizers()

for word in ["slowly", "happiness"]:
    for name, tok in tokenizers.items():
        out = tokenize_with_offsets(tok, word)
        print(f"{word:>10} {name:>5}: {out['tokens']}  offsets={out['offsets']}")

## 3. TODO 1: implement `boundary_aligned`

Return `True` if some token starts exactly at `boundary`.

For example, the boundary in `slow|ly` is 4. GPT-2's offsets are `[(0, 4), (4, 6)]`, so a token starts at 4 and the word is aligned.

The checks below pass once your function is right.

In [ ]:
def boundary_aligned(offsets, boundary):
    """offsets: list of (start, end) per token. boundary: where the suffix starts."""
    # TODO: replace this line
    raise NotImplementedError


assert boundary_aligned([(0, 4), (4, 6)], 4)      # slow | ly        -> aligned
assert not boundary_aligned([(0, 6)], 4)          # slowly (1 token) -> not aligned
assert not boundary_aligned([(0, 1), (1, 9)], 5)  # h | appiness     -> not aligned (suffix "ness" starts at 5)
print("All checks passed!")

## 4. TODO 2: run the experiment

For every sampled word and both tokenizers, record whether the boundary aligned and how many tokens the word used.

In [ ]:
rows = []
for _, row in sample.iterrows():
    for name, tok in tokenizers.items():
        out = tokenize_with_offsets(tok, row["target"])
        rows.append({
            "word": row["target"],
            "group": row["group"],
            "tokenizer": name,
            "tokens": out["tokens"],
            "aligned": ...,   # TODO: use boundary_aligned(...) and row["suffix_boundary"]
            "n_tokens": ...,  # TODO: how many tokens did this word use?
        })

results = pd.DataFrame(rows)
results.head()

## 5. Compare the groups

`aligned` is the fraction of words where a token starts at the suffix boundary. `n_tokens` is the average number of tokens per word.

In [ ]:
summary = results.groupby(["tokenizer", "group"])[["aligned", "n_tokens"]].mean().round(3)
summary

## 6. TODO 3: plot it

Make one plot comparing the alignment rate of the two groups for each tokenizer. A grouped bar chart works well (hint: `summary["aligned"].unstack().plot.bar()`). Label the axes and add a title.

In [ ]:
# TODO: make your plot here


plt.savefig("alignment.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Look at the misses

The averages are easier to interpret once you've seen real examples. Below are some rewritten words where the boundary was missed. Use one of them in `findings.md`.

In [ ]:
misses = results[(results["group"] == "rewritten") & ~results["aligned"]]
misses[["word", "tokenizer", "tokens"]].sample(10, random_state=0)

## 8. Write it up

Fill in `findings.md` with three short answers. Then follow **Submitting** in the README.